In [1]:
# Import required libraries

import shutil
import os
import h5py
import numpy as np
import pandas as pd
from tqdm import tqdm

In [2]:
# BASE_DIR = parent directory of the current working directory (project root)

BASE_DIR = os.path.dirname(os.path.dirname(os.getcwd()))
BASE_DIR

'd:\\CS406_FINAL_PROJECT'

In [3]:
# DATA_DIR = data directory

DATA_DIR = os.path.join(BASE_DIR, 'data')
DATA_DIR

'd:\\CS406_FINAL_PROJECT\\data'

In [4]:
# PROCESSED_IMG_DIR = directory that contains processed PNG images

PROCESSED_IMG_DIR = os.path.join(DATA_DIR, 'processed')
PROCESSED_IMG_DIR

'd:\\CS406_FINAL_PROJECT\\data\\processed'

In [5]:
# INPUT_DIRS = input directories containing .mat MRI files

INPUT_DIRS = [
    os.path.join(DATA_DIR, 'raw', 'brainTumorDataPublic_1-766'),
    os.path.join(DATA_DIR, 'raw', 'brainTumorDataPublic_767-1532'),
    os.path.join(DATA_DIR, 'raw', 'brainTumorDataPublic_1533-2298'),
    os.path.join(DATA_DIR, 'raw', 'brainTumorDataPublic_2299-3064'),
]
INPUT_DIRS

['d:\\CS406_FINAL_PROJECT\\data\\raw\\brainTumorDataPublic_1-766',
 'd:\\CS406_FINAL_PROJECT\\data\\raw\\brainTumorDataPublic_767-1532',
 'd:\\CS406_FINAL_PROJECT\\data\\raw\\brainTumorDataPublic_1533-2298',
 'd:\\CS406_FINAL_PROJECT\\data\\raw\\brainTumorDataPublic_2299-3064']

In [6]:
# CVIND_PATH = cvind.mat path that contains the 5-fold cross-validation indices

CVIND_PATH = os.path.join(DATA_DIR, 'raw', 'cvind.mat')
CVIND_PATH

'd:\\CS406_FINAL_PROJECT\\data\\raw\\cvind.mat'

In [7]:
# KFOLD_DIR = directory that contains 5-fold cross-validation subset

KFOLD_DIR = os.path.join(DATA_DIR, 'kfold_dataset')
if not os.path.exists(KFOLD_DIR):
    os.makedirs(KFOLD_DIR)
KFOLD_DIR

'd:\\CS406_FINAL_PROJECT\\data\\kfold_dataset'

In [8]:
# Mapping numeric labels to tumor names

CLASS_NAMES = {
    1: 'Meningioma Tumor', 
    2: 'Glioma Tumor', 
    3: 'Pituitary Tumor'
}
CLASS_NAMES

{1: 'Meningioma Tumor', 2: 'Glioma Tumor', 3: 'Pituitary Tumor'}

In [9]:
# Extract image's fold from cvind.mat

f = h5py.File(CVIND_PATH, 'r')
cvind = f['cvind'][:].T.flatten().astype(int)
cvind

array([5, 5, 5, ..., 4, 2, 1], shape=(3064,))

In [10]:
# Store the absolute paths of all .mat files

all_mat_files = []
for directory in INPUT_DIRS:
    files = [os.path.join(directory, f) for f in os.listdir(directory) if f.endswith('.mat')]
    all_mat_files.extend(files)
print(all_mat_files)
print(len(all_mat_files))

['d:\\CS406_FINAL_PROJECT\\data\\raw\\brainTumorDataPublic_1-766\\1.mat', 'd:\\CS406_FINAL_PROJECT\\data\\raw\\brainTumorDataPublic_1-766\\10.mat', 'd:\\CS406_FINAL_PROJECT\\data\\raw\\brainTumorDataPublic_1-766\\100.mat', 'd:\\CS406_FINAL_PROJECT\\data\\raw\\brainTumorDataPublic_1-766\\101.mat', 'd:\\CS406_FINAL_PROJECT\\data\\raw\\brainTumorDataPublic_1-766\\102.mat', 'd:\\CS406_FINAL_PROJECT\\data\\raw\\brainTumorDataPublic_1-766\\103.mat', 'd:\\CS406_FINAL_PROJECT\\data\\raw\\brainTumorDataPublic_1-766\\104.mat', 'd:\\CS406_FINAL_PROJECT\\data\\raw\\brainTumorDataPublic_1-766\\105.mat', 'd:\\CS406_FINAL_PROJECT\\data\\raw\\brainTumorDataPublic_1-766\\106.mat', 'd:\\CS406_FINAL_PROJECT\\data\\raw\\brainTumorDataPublic_1-766\\107.mat', 'd:\\CS406_FINAL_PROJECT\\data\\raw\\brainTumorDataPublic_1-766\\108.mat', 'd:\\CS406_FINAL_PROJECT\\data\\raw\\brainTumorDataPublic_1-766\\109.mat', 'd:\\CS406_FINAL_PROJECT\\data\\raw\\brainTumorDataPublic_1-766\\11.mat', 'd:\\CS406_FINAL_PROJECT\\da

In [11]:
# Sort all .mat files in ascending order

all_mat_files.sort(key=lambda x: int(os.path.splitext(os.path.basename(x))[0]))
print(all_mat_files)
print(len(all_mat_files))

['d:\\CS406_FINAL_PROJECT\\data\\raw\\brainTumorDataPublic_1-766\\1.mat', 'd:\\CS406_FINAL_PROJECT\\data\\raw\\brainTumorDataPublic_1-766\\2.mat', 'd:\\CS406_FINAL_PROJECT\\data\\raw\\brainTumorDataPublic_1-766\\3.mat', 'd:\\CS406_FINAL_PROJECT\\data\\raw\\brainTumorDataPublic_1-766\\4.mat', 'd:\\CS406_FINAL_PROJECT\\data\\raw\\brainTumorDataPublic_1-766\\5.mat', 'd:\\CS406_FINAL_PROJECT\\data\\raw\\brainTumorDataPublic_1-766\\6.mat', 'd:\\CS406_FINAL_PROJECT\\data\\raw\\brainTumorDataPublic_1-766\\7.mat', 'd:\\CS406_FINAL_PROJECT\\data\\raw\\brainTumorDataPublic_1-766\\8.mat', 'd:\\CS406_FINAL_PROJECT\\data\\raw\\brainTumorDataPublic_1-766\\9.mat', 'd:\\CS406_FINAL_PROJECT\\data\\raw\\brainTumorDataPublic_1-766\\10.mat', 'd:\\CS406_FINAL_PROJECT\\data\\raw\\brainTumorDataPublic_1-766\\11.mat', 'd:\\CS406_FINAL_PROJECT\\data\\raw\\brainTumorDataPublic_1-766\\12.mat', 'd:\\CS406_FINAL_PROJECT\\data\\raw\\brainTumorDataPublic_1-766\\13.mat', 'd:\\CS406_FINAL_PROJECT\\data\\raw\\brainTumo

In [12]:
# Store metadata for each image

data = []
for idx, mat_path in enumerate(tqdm(all_mat_files)):
    base_name = os.path.splitext(os.path.basename(mat_path))[0]

    png_path = os.path.join(PROCESSED_IMG_DIR, f'{base_name}.png')

    with h5py.File(mat_path, 'r') as f:
        label = int(f['cjdata']['label'][0][0])

        fold_id = cvind[idx]

        data.append({
            'src_path': png_path,
            'label_id': label,
            'fold': fold_id,
            'file_name': f'{base_name}.png'
        })

100%|██████████| 3064/3064 [00:02<00:00, 1400.41it/s]


In [13]:
# Create a Pandas DataFrame from the list of the metadata dictionaries

df = pd.DataFrame(data)
df.head()

,src_path,label_id,fold,file_name
0,d:\CS406_FINAL_PROJECT\data\processed\1.png,1,5,1.png
1,d:\CS406_FINAL_PROJECT\data\processed\2.png,1,5,2.png
2,d:\CS406_FINAL_PROJECT\data\processed\3.png,1,5,3.png
3,d:\CS406_FINAL_PROJECT\data\processed\4.png,1,5,4.png
4,d:\CS406_FINAL_PROJECT\data\processed\5.png,1,5,5.png


In [14]:
# Iterate through the DataFrame to organize files into a K-Fold directory structure

for _, row in tqdm(df.iterrows()):
    fold = row['fold']
    label_id = row['label_id']
    src_path = row['src_path']
    file_name = row['file_name']
    
    subset_name = f'Subset_{fold}'
    class_name = CLASS_NAMES[label_id]
    
    dest_dir = os.path.join(KFOLD_DIR, subset_name, class_name)
    if not os.path.exists(dest_dir):
        os.makedirs(dest_dir)
        
    shutil.copy2(src_path, os.path.join(dest_dir, file_name))

3064it [00:04, 654.00it/s]
